# 23 · WGFMU wake-up · L1 空连 (real B1500, no DUT)

> ⚠️ **执行前必读** ⚠️
>
> 此 notebook 连真机，**严禁接器件**。验证 wake-up 流水线整体跑通。
>
> 必须先跑过：
> - [x] 20_wgfmu_iv_sweep_dryrun (PASS)
> - [x] 22_wgfmu_wakeup_dryrun (PASS)
> - [x] 21_wgfmu_iv_sweep_realdevice (PASS, 开路 |I| < 1 µA)

**参数特意压到极低**（v_pgm=-1V, v_ers=+1V, 5 cycles），即便误接器件也不会写入。

**通过标准**：
- 无 dll 错
- `complete == total`
- `qc.status == "ok"`
- 5 个 read 窗都有数据
- `|i_read_mean| < 1 uA` (开路)


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt

print("Python:", sys.version.split()[0])
print("project root:", ROOT)


## 1. VISA + dll 自检


In [ ]:
import pyvisa

VISA_ADDR = "GPIB0::17::INSTR"  # 跟 21 用同一个，21 跑通这里就直接用

rm = pyvisa.ResourceManager()
print("resources:", rm.list_resources())
if VISA_ADDR not in rm.list_resources():
    raise RuntimeError(f"{VISA_ADDR} 不在资源列表，请检查或改地址")
rm.close()
print(f"✅ {VISA_ADDR} OK")


In [ ]:
from fefetlab.measurements.wgfmu.real_backend import RealWgfmuBackend

backend = RealWgfmuBackend()
backend.load()
print("✅ wgfmu.dll loaded")

backend.open_session(VISA_ADDR)
backend.set_timeout(60.0)
channel_ids = backend.get_channel_ids()
print(f"channels: {channel_ids}")

CHAN_ID = 101
assert CHAN_ID in channel_ids, f"CH{CHAN_ID} not detected"


## 2. 极小 wake-up: 1 stage × 5 cycles, ±1V

注意：v_pgm=-1V / v_ers=+1V **远低于 FeFET 切换电压**——即便误接器件也不会被烧/写。


In [ ]:
from fefetlab.measurements.wgfmu.wakeup import (
    WakeupStage, WakeupReadout, WgfmuWakeupConfig, WgfmuWakeupRunner
)

stages = [
    WakeupStage(
        n_cycles=5,
        v_pgm=-1.0,   # 安全幅值
        v_ers=+1.0,
        t_pgm_s=10e-6,
        t_ers_s=10e-6,
        rise_fall_s=1e-6,
        inter_pulse_s=2e-6,
        label="aircheck_1V",
    ),
]
readout = WakeupReadout(
    v_read=-0.3,
    t_read_s=5e-6,
    rise_fall_s=1e-6,
    measure_points=10,
    measure_average_s=200e-9,
)
cfg = WgfmuWakeupConfig(
    label="L1_aircheck_wakeup",
    chan_id=CHAN_ID,
    v_init=0.0,
    v_base=0.0,
    operation_mode="FASTIV",
    measure_mode="CURRENT",
    measure_current_range="1UA",   # 高灵敏度
    treat_warning_as_error=False,
    timeout_s=30.0,
    notes="L1 空连, 5 cycle, ±1V 安全幅值",
)

print(f"stages: {len(stages)}, total cycles: 5")
print(f"expected segments: 15 (5 cycles × 3)")


In [ ]:
runner = WgfmuWakeupRunner(backend=backend)
result = runner.run(
    resource=VISA_ADDR,
    stages=stages,
    readout=readout,
    cfg=cfg,
)
print(f"✅ run() returned")
print(f"complete / total : {result.complete} / {result.total}")
print(f"issues           : {result.issues}")


## 3. QC + cycle 数据


In [ ]:
print("=== qc_df ===")
print(result.qc_df.to_string(index=False))
print()
print("=== cycles_df ===")
cols = ["cycle_idx", "stage_label", "v_pgm", "v_ers", "v_read",
        "i_read_mean", "i_read_std", "n_samples"]
print(result.cycles_df[cols].to_string(index=False))


In [ ]:
err = result.meta.get("error_summary", "")
wrn = result.meta.get("warning_summary", "")
print("error_summary:")
print(err if err else "  (empty)")
print()
print("warning_summary:")
print(wrn if wrn else "  (empty)")


## 4. 画图: 波形 + per-cycle i_read


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 6))

# upper: 完整波形
t_wf, v_wf = result.plan.waveform_samples(dt_s=2e-7)
axes[0].plot(t_wf*1e6, v_wf, color="#1f77b4", lw=0.9)
axes[0].set_xlabel("time (us)")
axes[0].set_ylabel("voltage (V)")
axes[0].set_title("L1 aircheck · wake-up waveform (5 cycles, ±1V)")
axes[0].grid(alpha=0.3)
axes[0].axhline(0, color="gray", lw=0.5)
# 标 read 窗
for seg in result.plan.segments:
    if seg["measure_points"] > 0:
        axes[0].axvspan(seg["t_high_start_s"]*1e6, seg["t_high_end_s"]*1e6,
                        alpha=0.25, color="green")

# lower: per-cycle i_read
cyc = result.cycles_df
axes[1].errorbar(cyc["cycle_idx"], cyc["i_read_mean"]*1e9,
                 yerr=cyc["i_read_std"]*1e9, fmt="o-", color="#d62728")
axes[1].set_xlabel("cycle index")
axes[1].set_ylabel("i_read_mean (nA)")
axes[1].set_title("per-cycle readout current (open-circuit → should be ~0)")
axes[1].grid(alpha=0.3)
axes[1].axhline(0, color="gray", lw=0.5)

plt.tight_layout()
plt.show()

i_abs_max = cyc["i_read_mean"].abs().max()
print(f"\n|i_read_mean|_max = {i_abs_max*1e9:.3f} nA")
if i_abs_max < 1e-6:
    print("✅ 开路验证通过")
else:
    print(f"⚠️  |i_read_mean| 偏大 ({i_abs_max*1e6:.3f} uA) — 检查探针/接地")


## 5. 落盘路径


In [ ]:
print("output paths:")
for k, v in result.paths.items():
    print(f"  {k:20s} -> {v}")


---

## 通过标准回顾

- [ ] cell 1-2: VISA + dll OK
- [ ] cell 3-4: `complete == total`，cycles_df 5 行齐全
- [ ] cell 5: 错误/警告均为空
- [ ] cell 6: 波形看到 5 个 cycle 完整结构，`|i_read|_max < 1 uA`
- [ ] cell 7: 路径打印正常

**L1 全部 ✅ → 阿耶会写 24/25 真器件 notebook，那时 yhzang 才接器件**
